# Datums done properly: geodetic ↔ State Plane (Rochester, NY)

**The everyday problem.** The rover reports latitude/longitude and an ellipsoid height.
The plat, the county GIS, and the deed all live somewhere else: **NAD83 State Plane
grid coordinates in US survey feet**, with elevations on **NAVD88**. Converting between
those worlds is not a unit conversion — it is a datum realisation, a map projection,
and a scale change, and each step moves the numbers by amounts that matter at survey
precision.

This notebook works two control points in **Rochester, NY (NY West zone, EPSG:2262)**
end to end with `pyproj`:

1. WGS84 → NAD83(2011): the same physical point, two datums, a real coordinate shift.
2. NAD83(2011) geographic → State Plane grid coordinates (US survey feet).
3. Grid distance vs ground (geodetic) distance, convergence, and point scale factor —
   the reason a 1 km line is *not* 1 km on the grid.
4. Round-trip the conversion and measure the closure (it should be millimetres).
5. Heights: ellipsoid vs orthometric, and the one mistake to never make.

In [1]:
from pyproj import CRS, Proj, Transformer, Geod

FTUS = 1200.0 / 3937.0   # exact US survey foot, metres

# Two control points, downtown Rochester, NY (WGS84 geographic)
P1 = {'name': 'P1 — Exchange & Broad',  'lat': 43.156600, 'lon': -77.608800}
P2 = {'name': 'P2 — NE Rochester',      'lat': 43.168300, 'lon': -77.595500}

wgs84   = CRS.from_epsg(4326)
nad83   = CRS.from_epsg(6318)   # NAD83(2011), geographic
spc     = CRS.from_epsg(2262)   # NAD83 / New York West, US survey feet
print('State Plane CRS:', spc.name)
print('Axis units     :', spc.axis_info[0].unit_name)

State Plane CRS: NAD83 / New York West (ftUS)
Axis units     : US survey foot


## 1 · Same point, two datums — and a warning about "WGS84"

WGS84 and NAD83(2011) are different datums: NAD83 is fixed to the North American plate,
WGS84 tracks the global frame, and in New York the two currently differ by roughly a
metre or two *when the WGS84 coordinates carry a modern realisation and epoch*
(e.g. a 2026-epoch CORS solution). But watch what the library does below: a plain
EPSG:4326 → EPSG:6318 transform applies PROJ's **ballpark** transformation and reports
a shift of exactly zero — PROJ treats generic "WGS84" as coincident with NAD83.

That zero is the lesson. "WGS84" without a realisation and an epoch is not a
survey-grade statement, and a silent ballpark shift is how metre-level errors get
into plats. For centimetre work, transform with realised frames and epochs
(NGS NCAT, or your controller's datum settings) — never with a bare CRS pair.

In [2]:
t_datum = Transformer.from_crs(wgs84, nad83, always_xy=True)
for p in (P1, P2):
    lon83, lat83 = t_datum.transform(p['lon'], p['lat'])
    p['lon83'], p['lat83'] = lon83, lat83
    print(f"{p['name']}: WGS84 ({p['lat']:.6f}, {p['lon']:.6f})  ->  "
          f"NAD83(2011) ({lat83:.6f}, {lon83:.6f})")
    print(f"   shift: dLat {(lat83-p['lat'])*3600:+.2f}\"  dLon {(lon83-p['lon'])*3600:+.2f}\"")

P1 — Exchange & Broad: WGS84 (43.156600, -77.608800)  ->  NAD83(2011) (43.156600, -77.608800)
   shift: dLat +0.00"  dLon +0.00"
P2 — NE Rochester: WGS84 (43.168300, -77.595500)  ->  NAD83(2011) (43.168300, -77.595500)
   shift: dLat +0.00"  dLon +0.00"


## 2 · Geographic → State Plane grid

EPSG:2262 (New York West) is a Transverse Mercator projection — central meridian
78°35′ W, scale factor at origin 0.9999375 — in **US survey feet**
(1 ftUS = 1200/3937 m exactly — *not* the international foot; the 2 ppm difference
is a classic blunder source on long lines).

In [3]:
t_spc = Transformer.from_crs(nad83, spc, always_xy=True)
for p in (P1, P2):
    E_ft, N_ft = t_spc.transform(p['lon83'], p['lat83'])
    p['E_ft'], p['N_ft'] = E_ft, N_ft
    print(f"{p['name']}:  E {E_ft:,.3f} ftUS   N {N_ft:,.3f} ftUS"
          f"   (= {E_ft*FTUS:,.3f} m, {N_ft*FTUS:,.3f} m)")

P1 — Exchange & Broad:  E 1,408,322.595 ftUS   N 1,151,662.281 ftUS   (= 429,257.586 m, 351,027.365 m)
P2 — NE Rochester:  E 1,411,821.108 ftUS   N 1,155,968.129 ftUS   (= 430,323.934 m, 352,339.790 m)


## 3 · Grid distance ≠ ground distance

A distance measured on the ground must be **reduced to the ellipsoid** (elevation
factor) and then **scaled by the projection** (point scale factor) before it can be
compared with the inverse distance between grid coordinates. `pyproj` exposes both
the meridian convergence (grid north vs true north) and the point scale factor:

In [4]:
f = Proj(spc).get_factors(P1['lon83'], P1['lat83'])
print(f"At P1:  meridian convergence = {f.meridian_convergence:+.6f}° "
      f"({f.meridian_convergence*3600:+.1f} seconds)")
print(f"        point scale factor   = {f.tissot_semimajor:.8f} "
      f"({(f.tissot_semimajor-1)*1e6:+.1f} ppm)")

dE = (P2['E_ft'] - P1['E_ft']) * FTUS
dN = (P2['N_ft'] - P1['N_ft']) * FTUS
grid_dist = (dE**2 + dN**2) ** 0.5

geod = Geod(ellps='GRS80')
_, _, ground_dist = geod.inv(P1['lon83'], P1['lat83'], P2['lon83'], P2['lat83'])
print(f"\nGrid inverse distance    : {grid_dist:.4f} m")
print(f"Geodetic (ground) distance: {ground_dist:.4f} m")
print(f"Difference: {(grid_dist-ground_dist)*1000:+.1f} mm  "
      f"({(grid_dist/ground_dist-1)*1e6:+.1f} ppm combined effect)")

At P1:  meridian convergence = +0.666610° (+2399.8 seconds)
        point scale factor   = 1.00001475 (+14.7 ppm)

Grid inverse distance    : 1691.0229 m
Geodetic (ground) distance: 1690.9962 m
Difference: +26.7 mm  (+15.8 ppm combined effect)


Fifteen parts per million sounds like nothing until it is 16 mm on every kilometre —
several times the precision of the instrument that measured it. This is why state-plane
work is always labelled *grid* or *ground*, and why a combined factor belongs on the
face of the plat.

In [5]:
# 4 · Round-trip closure: grid -> geographic must return the input
t_back = Transformer.from_crs(spc, nad83, always_xy=True)
for p in (P1, P2):
    lon_rt, lat_rt = t_back.transform(p['E_ft'], p['N_ft'])
    err_m = (((lon_rt - p['lon83']) * 111320 * 0.8) ** 2 +
             ((lat_rt - p['lat83']) * 110540) ** 2) ** 0.5
    print(f"{p['name']}: round-trip closure ≈ {err_m*1000:.2f} mm")

P1 — Exchange & Broad: round-trip closure ≈ 0.00 mm
P2 — NE Rochester: round-trip closure ≈ 0.00 mm


## 5 · Heights: the mistake to never make

GNSS gives an **ellipsoid height** `h` (above the GRS80 ellipsoid). Plats and flood
certificates need an **orthometric height** `H` (above the geoid — "sea level"):

$$H = h - N$$

where `N` is the **geoid height** from a model such as **GEOID18** (in western New York
`N` is roughly −34 m, so the two heights differ by about 34 m, not by a rounding error).

The trap: a coordinate library will happily "convert" an ellipsoid height to NAVD88
by changing only the *units* (metres → feet) if the geoid grid is unavailable —
producing a number that looks precise and is wrong by the full geoid height.
The defensible routes are the NGS NCAT tool, a controller running the current
GEOID model, or a transformation library *verified* to have the grid loaded.
Always state the geoid model and epoch on the deliverable.

---
*surveyor-brain — worked notebooks from field practice. Synthetic-but-realistic data;
methods are the real thing.*